langGraph的另一套api,主要用来直接在python中进行编排而不是使用图结构进行编排


流程拆解这个智能体，1查询天气定义tools,询问大模型返回结果

In [19]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

# 环境变量引入
load_dotenv()

model=init_chat_model(
        model="openai:qwen3.6-plus-2026-04-02",
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url=os.getenv("OPENAI_BASE_URL"),
        temperature=0,
)

In [20]:
import requests
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """查询天气（使用免费 API）"""

    # 1. 先把城市转经纬度（简单写死示例）
    geo_map = {
        "北京": (39.90, 116.40),
        "上海": (31.23, 121.47),
        "深圳": (22.54, 114.06),
        "成都": (22.54, 114.06),
    }

    if city not in geo_map:
        return f"没有找到 {city} 的天气数据"

    lat, lon = geo_map[city]

    # 2. 免费天气 API（Open-Meteo）
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"

    res = requests.get(url).json()

    weather = res["current_weather"]

    return f"{city} 当前温度 {weather['temperature']}°C，风速 {weather['windspeed']}km/h"


# 绑定工具
tools=[get_weather]
# 做模型名称映射
tools_name={tool.name:tool for tool in tools}
model_with_tools=model.bind_tools(tools)

In [21]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langgraph.func import task


#定义询问模型的任务

@task
def call_llm(messages):
    response=model_with_tools.invoke([
        SystemMessage(content="你是天气助手，需要时必须调用工具")
        ]+messages
    )

    return response


In [22]:
from json import tool

from langgraph.func import entrypoint, task


@entrypoint()
def  main(messages):
     #询问大模型我要查询天气
     model_response=call_llm(messages).result()

# 查询天气的agent     # 2️⃣ Agent loop（核心）

     while True:
           # 判断没有工具调用直接结束
           if not model_response.tool_calls:
               print("没有没有工具")
               return  model_response
           #定义一个工具消息集合
           tool_messages = []
           for tool_call in model_response.tool_calls:
                # 1. 工具名
                tool_name = tool_call["name"]
                # 2. 工具参数
                tool_args = tool_call["args"]
                #3函数
                tool=tools_name[tool_name]
                result = tool.invoke(tool_args)
                print('这是工具调用返回后的结果',tool_call)
                print('这是工具调用返回后的结果',result)
                tool_messages.append(ToolMessage(content=str(result),
                                                 tool_call_id=tool_call["id"]))
           # 先临时 return，避免死循环
           return model_response
main.invoke([HumanMessage(content='成都今天天气怎么样')])

APIConnectionError: Connection error.